In [ ]:
# load the names dataset from file
from pathlib import Path

data_dir = Path.cwd() / ".." / ".." / "data"


def load_names(path: Path) -> list[str]:
    with path.open("r") as f:
        return f.read().splitlines()


words = load_names(data_dir / "names.txt")
words[:4]

**Create the Training Set**

In [ ]:
# global sentinel tokens (start and stop)
TOKEN_DOT = "."
# the vocabulary size
VOCAB_SIZE = 27

In [ ]:
# LUT construction
chars = sorted(list(set("".join(words))))

# string-to-index
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi[TOKEN_DOT] = 0

# index to string
itos = {i: c for c, i in stoi.items()}

assert len(stoi) == len(itos), "broken invariant"
assert all(itos[stoi[c]] == c for c in chars), "broken invariant"

In [ ]:
# create the training set (all bigrams)
import torch

xs_raw, ys_raw = [], []
for w in words[:1]:
    chs = [TOKEN_DOT] + list(w) + [TOKEN_DOT]
    for l, r in zip(chs, chs[1:]):
        xs_raw.append(stoi[l])
        ys_raw.append(stoi[r])

xs = torch.tensor(xs_raw)
ys = torch.tensor(ys_raw)

**One-Hot Encoding**

In [ ]:
import torch.nn.functional as F

xenc = F.one_hot(xs, num_classes=VOCAB_SIZE).float()

**Our First Neuron**

In [ ]:
W = torch.randn((VOCAB_SIZE, VOCAB_SIZE), requires_grad=True)
xenc @ W

**Transforming Outputs**

In [ ]:
logits = xenc @ W  # log counts
counts = logits.exp()  # equivalent to the N array from previous implementation
probs = counts / counts.sum(
    axis=1, keepdim=True
)  # normalize to probability distribution, as before

**Checking Current Performance**

**Vectorized Loss**

In [ ]:
loss = -probs[torch.arange(5), ys].log().mean()
loss

**Backward Pass & Update**

In [ ]:
W.grad = None  # zero the gradient
loss.backward()

In [ ]:
# update
W.data += -0.1 * W.grad

**Putting it Together**

In [ ]:
import torch

xs_raw, ys_raw = [], []
for w in words:
    chs = [TOKEN_DOT] + list(w) + [TOKEN_DOT]
    for l, r in zip(chs, chs[1:]):
        xs_raw.append(stoi[l])
        ys_raw.append(stoi[r])

xs = torch.tensor(xs_raw)
ys = torch.tensor(ys_raw)

In [ ]:
# initialize the network
W = torch.randn((VOCAB_SIZE, VOCAB_SIZE), requires_grad=True)

In [ ]:
for k in range(1024):
    # encoding
    xenc = F.one_hot(xs, num_classes=VOCAB_SIZE).float()

    # forward pass
    logits = xenc @ W
    counts = logits.exp()
    probs = counts / counts.sum(axis=1, keepdim=True)

    # loss
    loss = -probs[torch.arange(xs.nelement()), ys].log().mean()

    # backward pass
    W.grad = None
    loss.backward()

    # update
    lr = 50 / (1 + 2 * k)
    W.data += -lr * W.grad

# final training loss
print(loss.item())